# Kaggle: three ML models, five repeated stratified splits

For seeds **42, 123, 2026, 3407, 777**, split the full data 80/20, then reserve 20% of the 80% for validation: **64% train / 16% validation / 20% test**. LightGBM, XGBoost and Random Forest share the same row indices within each run. Complete all three models before advancing to the next seed.

This is **repeated stratified holdout**, not five-fold CV. There is no additional CV loop. Test sets may overlap between runs. Freeze model settings before executing; test results must not drive retuning.

Preprocessing is fitted using each run's training rows only. Thresholds are selected using its validation rows only. Save the same model's test results **before tuning (argmax)** and **after tuning**, plus paired changes. Summaries report mean ± sample standard deviation across five test runs; these are descriptive variability measures, not confidence intervals.

Upload one full CSV or a ZIP containing one full CSV. If using your older ZIP containing train/test CSVs, explicitly set COMBINE_EXISTING_SPLITS=True to reunite them into the full pool and replace the old split. Both files must contain Severity labels. This changes the evaluation protocol and is not directly comparable with your earlier fixed-test results.

XGBoost requires CUDA; sklearn Random Forest uses CPU; LightGBM defaults to CPU. Full data are used without a sample cap. Five seeds require 15 full model fits; Kaggle time/RAM completion has not been measured.

Run the installation cell, set the dataset path, then Run All. Results and charts are saved in a unique folder under /kaggle/working.
All four Severity levels (1–4) are required in the full dataset and every partition. All five splits are checked before training. Each run displays and saves `split_severity_counts.csv`. If a level is missing, execution stops with an error; classes are never fabricated.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','lightgbm','xgboost','scikit-learn>=1.4','pandas','numpy','scipy','joblib','matplotlib'],check=True)


In [ ]:
import json, os, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import scipy.sparse as sp
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,
                             precision_recall_curve,classification_report,confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight

import gc, hashlib, time, zipfile, shutil
from datetime import datetime, timezone
from IPython.display import display, FileLink
import matplotlib.pyplot as plt

# EDIT: the full CSV or ZIP in your attached Kaggle dataset.
DATA_PATH = Path('/kaggle/input/YOUR_DATASET/US_Accidents.csv')
# For a ZIP with multiple unrelated CSVs, select a single full-data CSV member.
CSV_MEMBER = None
# Only enable this to recombine your previously split train and test files.
COMBINE_EXISTING_SPLITS = False

SEEDS = [42, 123, 2026, 3407, 777]
RANDOM_STATE = SEEDS[0]  # Default helper seed; overridden inside every run.
TARGET = 'Severity'
REQUIRED_SEVERITY_LEVELS = [1, 2, 3, 4]
LGB_DEVICE = 'cpu'
XGB_DEVICE = 'cuda'
USE_CLASS_WEIGHT = True
CLASS_WEIGHT_CAP = 2.5
MAX_CAT_ONEHOT = 50
THRESHOLD_BETA = 1.0
SAVE_MODELS = False  # Enable if you need all 15 fitted model files.
SAVE_TEST_PREDICTIONS = True
SAVE_TEST_PROBABILITIES = False  # Enable for later analysis; increases disk usage.
SAVE_SPLIT_INDICES = True
DATETIME_COLS = ['Start_Time','End_Time','Weather_Timestamp']
BOOL_COLS = ['Amenity','Crossing','Give_Way','Junction','No_Exit','Railway','Station','Stop','Traffic_Signal']
GEO_COLS = ['Start_Lat','Start_Lng']
YELLOW_EXCLUDE = ['ID','Source','End_Lat','End_Lng','Country','Wind_Chill(F)',
                  'Precipitation(in)','Bump','Roundabout','Traffic_Calming','Turning_Loop']
METRICS = ['Accuracy','Macro_Precision','Macro_Recall','Macro_F1']
MODEL_NAMES = ['LightGBM','XGBoost','RandomForest']
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
OUTPUT_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUT_DIR = OUTPUT_ROOT / f'ml_repeated_splits_{RUN_ID}'
OUT_DIR.mkdir(parents=True,exist_ok=False)
assert len(SEEDS)==5 and len(set(SEEDS))==5, 'Use five distinct seeds.'
print('Output:',OUT_DIR)


In [ ]:
def check_compute():
    result=subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True)
    if result.returncode!=0: raise RuntimeError('Select a Kaggle GPU accelerator.')
    print(result.stdout.strip())
    probe=xgb.XGBClassifier(n_estimators=2,max_depth=2,tree_method='hist',device=XGB_DEVICE,
        objective='multi:softprob',num_class=3,eval_metric='mlogloss',random_state=SEEDS[0])
    probe.fit(np.arange(90,dtype=np.float32).reshape(30,3),np.tile(np.arange(3),10))
    actual=json.loads(probe.get_booster().save_config())['learner']['generic_param']['device']
    if not actual.startswith('cuda'): raise RuntimeError(f'XGBoost fell back to {actual}.')
    if LGB_DEVICE=='gpu':
        lgb.LGBMClassifier(n_estimators=2,min_child_samples=1,device_type='gpu',verbosity=-1).fit(
            np.arange(120,dtype=np.float32).reshape(40,3),np.tile(np.arange(4),10))
    print('XGBoost CUDA ready; Random Forest CPU; LightGBM:',LGB_DEVICE)
check_compute()


## Load the full dataset and preserve row identities

In [ ]:
def engineer_features(df):
    df = df.copy()
    df = df.drop(columns=YELLOW_EXCLUDE, errors="ignore")
    # End_Time occurs after an accident and must not define an input feature.
    df = df.drop(columns=["End_Time"], errors="ignore")
    for c in DATETIME_COLS:
        if c in df.columns: df[c] = pd.to_datetime(df[c], errors="coerce")
    if "Start_Time" in df.columns:
        df["Start_Hour"] = df["Start_Time"].dt.hour
        df["Start_Dayofweek"] = df["Start_Time"].dt.dayofweek
        df["Start_IsWeekend"] = (df["Start_Dayofweek"] >= 5).astype(int)
        df["Is_RushHour"] = df["Start_Hour"].isin([7,8,9,16,17,18]).astype(int)
    if "Weather_Timestamp" in df.columns:
        df["Weather_Hour"] = df["Weather_Timestamp"].dt.hour
    df = df.drop(columns=[c for c in DATETIME_COLS if c in df.columns], errors="ignore")
    for c in BOOL_COLS:
        if c in df.columns:
            if df[c].dtype == object:
                df[c] = df[c].map({"True":1,"False":0,True:1,False:0,"TRUE":1,"FALSE":0,"Y":1,"N":0})
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
    if "Weather_Condition" in df.columns:
        df["Weather_Condition"] = (df["Weather_Condition"].astype(str).str.lower().str.strip()
                                   .str.replace(r"[^a-z ]","",regex=True))
        # Frequency/target encoding learns its vocabulary from each training fold only.
    for c in df.columns:
        if df[c].dtype == object: df[c] = df[c].fillna("Missing").astype(str)
    df = df.dropna(subset=[TARGET])  # keep rows with missing coordinates
    return df

def load_full_data(path):
    if not path.is_file(): raise FileNotFoundError(f'Correct DATA_PATH: {path}')
    if zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as z:
            members=[x for x in z.namelist() if x.lower().endswith('.csv') and not x.startswith('__MACOSX/')]
            if CSV_MEMBER is not None:
                if CSV_MEMBER not in members: raise ValueError(f'CSV_MEMBER not found. Available: {members}')
                selected=[CSV_MEMBER]
            elif len(members)==1: selected=members
            elif COMBINE_EXISTING_SPLITS:
                train=[x for x in members if 'train' in Path(x).name.lower()]
                test=[x for x in members if 'test' in Path(x).name.lower()]
                if len(train)!=1 or len(test)!=1 or train[0]==test[0]:
                    raise ValueError(f'Expected one train and one test member: {members}')
                selected=[train[0],test[0]]
            else: raise ValueError(f'ZIP has multiple CSVs: {members}. Set CSV_MEMBER or enable COMBINE_EXISTING_SPLITS.')
            frames=[]; identity=[]
            for member in selected:
                with z.open(member) as f: frame=pd.read_csv(f,low_memory=False)
                if TARGET not in frame: raise ValueError(f'{member} needs {TARGET}.')
                if frames and set(frame.columns)!=set(frames[0].columns): raise ValueError('CSV feature schemas differ.')
                frames.append(frame)
                info=z.getinfo(member)
                identity.append({'member':member,'crc32':f'{info.CRC:08x}','bytes':info.file_size,'rows':len(frame)})
            return pd.concat(frames,ignore_index=True),identity
    if path.suffix.lower() not in ['.csv','.gz']: raise ValueError('Use CSV, gzip CSV, or a ZIP of CSV files.')
    h=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''):h.update(block)
    return pd.read_csv(path,low_memory=False),{'sha256':h.hexdigest(),'file':path.name}

raw, source_identity=load_full_data(DATA_PATH)
raw=raw.reset_index(drop=True)
original_rows=len(raw)
df=engineer_features(raw)
del raw;gc.collect()
# Feature engineering retains original indices; labels missing from Severity are excluded.
row_ids=df.index.to_numpy(dtype=np.int64)
labels=pd.to_numeric(df[TARGET],errors='raise')
if not np.isfinite(labels).all() or not labels.isin([1,2,3,4]).all():
    raise ValueError('Severity must contain only 1,2,3,4 after removing missing labels.')
classes=sorted(labels.astype(int).unique().tolist())
missing_levels=sorted(set(REQUIRED_SEVERITY_LEVELS)-set(classes))
if missing_levels:
    raise ValueError(f'Full dataset is missing Severity levels {missing_levels}. All four levels are required; splitting cannot create missing classes.')
class_map={int(c):i for i,c in enumerate(classes)}
y=labels.astype(int).map(class_map).to_numpy(dtype=np.int64)
X=df.drop(columns=[TARGET]);del df;gc.collect()
n_classes=len(classes)
if n_classes<2: raise ValueError('At least two classes are required.')
counts=pd.Series(y).value_counts()
if counts.min()<10: raise ValueError('Not enough rows in a class for stable stratified 64/16/20 splitting.')
print('Full pool:',len(X),'| excluded missing-label rows:',original_rows-len(X),'| classes:',class_map)
protocol={'method':'five repeated stratified holdouts','seeds':SEEDS,'fractions':{'train':.64,'validation':.16,'test':.20},
 'source':source_identity,'original_rows':original_rows,'usable_rows':len(X),'class_map':class_map,
 'LGB_DEVICE':LGB_DEVICE,'XGB_DEVICE':XGB_DEVICE,'class_weight_cap':CLASS_WEIGHT_CAP,
 'threshold_beta':THRESHOLD_BETA,'std_ddof':1,'combine_existing_splits':COMBINE_EXISTING_SPLITS}
(OUT_DIR/'protocol.json').write_text(json.dumps(protocol,indent=2))


## Training, validation threshold tuning, and paired test evaluation
Every run starts with fresh preprocessing and models. The same trained model produces both before- and after-tuning test predictions. Test labels are used only to compute reports. No parameters are selected from test results.

In [ ]:
def calc_metrics(y_true, y_pred):
    return {"Accuracy": accuracy_score(y_true, y_pred),
            "Macro_Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "Macro_Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "Macro_F1": f1_score(y_true, y_pred, average="macro", zero_division=0)}


THRESHOLD_BETA = 1.0

def tune_thresholds(y_val, proba_val, classes, beta=THRESHOLD_BETA):
    """Per-class decision thresholds maximizing F_beta on validation probabilities.
    Shared by LightGBM / XGBoost / RF / MLP / ST-GNN."""
    y_val = np.asarray(y_val); thresholds = {}
    for i, c in enumerate(classes):
        y_bin = (y_val == c).astype(int)
        prec, rec, thr = precision_recall_curve(y_bin, proba_val[:, i])
        thr = np.append(thr, 1.0)
        with np.errstate(divide="ignore", invalid="ignore"):
            fbeta = (1 + beta**2) * prec * rec / (beta**2 * prec + rec)
        fbeta = np.nan_to_num(fbeta)
        thresholds[c] = float(thr[np.argmax(fbeta)])
    return thresholds

def predict_with_thresholds(proba, thresholds, classes, fallback_class=0):
    """Apply per-class thresholds; samples passing no class fall back to fallback_class."""
    classes = np.array(classes)
    thr_arr = np.array([thresholds[c] for c in classes])
    passed = proba >= thr_arr[None, :]
    masked = np.where(passed, proba, -np.inf)
    preds = classes[masked.argmax(1)]
    preds[~passed.any(1)] = fallback_class
    return preds

def class_weight_dict(y, cap=CLASS_WEIGHT_CAP):
    """Balanced class weights as {class_label: weight}, capped so the ratio between the largest
    and smallest weight never exceeds `cap` (any weight above cap * min_weight is clipped down to
    that ceiling). This is the single source of truth for imbalance weighting - LightGBM, XGBoost,
    RandomForest (all via sample_weight), MLP (via oversampling below), and both GPU variants all
    derive their weighting from this same capped dict, so every model uses the identical ratio.
    Returns None if class weighting is off."""
    if not USE_CLASS_WEIGHT:
        return None
    classes = np.unique(y)
    raw_weights = compute_class_weight("balanced", classes=classes, y=y)
    min_w = raw_weights.min()
    capped_weights = np.minimum(raw_weights, min_w * cap)
    return dict(zip(classes, capped_weights))

def class_weights(y, cap=CLASS_WEIGHT_CAP):
    """Per-sample weights expanded from the capped class_weight_dict() above - used as
    `sample_weight` for LightGBM, XGBoost, and RandomForest (all trained via sample_weight rather
    than a "balanced"/"balanced_subsample" string, so the same cap applies to all three)."""
    cw = class_weight_dict(y, cap)
    if cw is None:
        return None
    y_arr = np.asarray(y)
    return np.array([cw[v] for v in y_arr])

def preprocess_ml(X_tr, y_tr, X_va, num=None, high=None, low=None, seed=RANDOM_STATE):
    num = list(numeric_cols) if num is None else list(num)
    high = list(high_card) if high is None else list(high)
    low = list(low_card) if low is None else list(low)
    X_tr, X_va = X_tr.copy(), X_va.copy()
    original_columns = list(X_tr.columns)
    freq_maps = {}
    for c in high:
        f = X_tr[c].value_counts(normalize=True)
        freq_maps[c] = f
        X_tr[c+"_freq"] = X_tr[c].map(f).fillna(0.0).astype(np.float32)
        X_va[c+"_freq"] = X_va[c].map(f).fillna(0.0).astype(np.float32)
        num.append(c+"_freq")
    X_tr, X_va = X_tr.drop(columns=high), X_va.drop(columns=high)
    try:
        from sklearn.preprocessing import TargetEncoder
        enc = TargetEncoder(random_state=seed)
    except Exception:
        from sklearn.preprocessing import OneHotEncoder
        enc = OneHotEncoder(handle_unknown="ignore", min_frequency=5)
    pre = ColumnTransformer([
        ("num", Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())]), num),
        ("cat", Pipeline([("i", SimpleImputer(strategy="most_frequent")), ("e", enc)]), low),
    ])
    X_tr_p = pre.fit_transform(X_tr, y_tr)
    pre.high_card_freq_maps_ = freq_maps
    pre.high_card_columns_ = high
    pre.input_columns_ = original_columns
    return X_tr_p, pre.transform(X_va), pre


def dense32(a):
    return np.asarray(a.toarray() if sp.issparse(a) else a, dtype=np.float32)

def make_models(seed=RANDOM_STATE):
    return {
        'LightGBM': lgb.LGBMClassifier(n_estimators=250, learning_rate=0.05,
            num_leaves=31, n_jobs=-1, verbosity=-1, device_type=LGB_DEVICE, random_state=seed),
        'XGBoost': xgb.XGBClassifier(n_estimators=250, max_depth=8, learning_rate=0.05,
            tree_method='hist', device=XGB_DEVICE, objective='multi:softprob', num_class=n_classes,
            eval_metric='mlogloss', n_jobs=-1, random_state=seed),
        'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=15,
            min_samples_leaf=20, max_features=0.3, n_jobs=-1, random_state=seed)
    }



In [ ]:
def split_for_seed(y, seed):
    """Stratified 64/16/20 split; reject any partition missing Severity 1–4."""
    y = np.asarray(y)
    expected = set(range(len(REQUIRED_SEVERITY_LEVELS)))
    if set(np.unique(y)) != expected:
        raise ValueError('Encoded labels must include all four Severity levels (1, 2, 3, 4).')
    positions = np.arange(len(y), dtype=np.int64)
    dev, test = train_test_split(positions, test_size=.20, stratify=y, random_state=seed)
    train, val = train_test_split(dev, test_size=.20, stratify=y[dev], random_state=seed)
    combined = np.concatenate([train, val, test])
    if len(combined) != len(y) or len(np.unique(combined)) != len(y):
        raise RuntimeError(f'Seed {seed}: partitions overlap or do not cover the full dataset.')
    for name, idx in [('Train', train), ('Validation', val), ('Test', test)]:
        missing = expected - set(np.unique(y[idx]))
        if missing:
            levels = [REQUIRED_SEVERITY_LEVELS[c] for c in sorted(missing)]
            raise ValueError(f'Seed {seed}: {name} is missing Severity {levels}. '
                             'Add more records for the rare classes before running training.')
    return train, val, test

def severity_counts_for_split(y, train_idx, val_idx, test_idx):
    """Counts use original Severity labels, not the encoded 0–3 labels."""
    table = pd.DataFrame({
        name: np.bincount(y[idx], minlength=len(REQUIRED_SEVERITY_LEVELS))
        for name, idx in [('Train', train_idx), ('Validation', val_idx), ('Test', test_idx)]
    }, index=pd.Index(REQUIRED_SEVERITY_LEVELS, name='Severity'))
    if not (table.to_numpy() > 0).all():
        raise ValueError('Every partition must contain at least one record from each Severity level.')
    return table

# Validate every planned seed before any expensive model training begins.
for seed in SEEDS:
    split_for_seed(y, seed)
print('Preflight passed: every seed includes Severity 1, 2, 3, and 4 in all three partitions.')


def report_predictions(model_name,stage,y_true,pred,run_dir,seed,run_number):
    pd.DataFrame(confusion_matrix(y_true,pred,labels=range(n_classes)),index=classes,columns=classes).to_csv(
        run_dir/f'{model_name}_{stage}_confusion_matrix.csv')
    report=pd.DataFrame(classification_report(y_true,pred,labels=range(n_classes),
        target_names=[str(c) for c in classes],output_dict=True,zero_division=0)).T
    report.to_csv(run_dir/f'{model_name}_{stage}_classification_report.csv')
    for label in classes:
        rec=report.loc[str(label)]
        per_class_rows.append({'Run':run_number,'Seed':seed,'Model':model_name,'Stage':stage,'Severity':label,
            'Precision':rec['precision'],'Recall':rec['recall'],'F1':rec['f1-score'],'Support':int(rec['support'])})

def flush_results():
    pd.DataFrame(metric_rows).to_csv(OUT_DIR/'run_test_metrics.csv',index=False)
    pd.DataFrame(delta_rows).to_csv(OUT_DIR/'run_threshold_deltas.csv',index=False)
    pd.DataFrame(per_class_rows).to_csv(OUT_DIR/'run_per_class_metrics.csv',index=False)

metric_rows=[];delta_rows=[];per_class_rows=[]
for run_number,seed in enumerate(SEEDS,1):
    run_dir=OUT_DIR/f'run_{run_number}_seed_{seed}';run_dir.mkdir()
    train_idx,val_idx,test_idx=split_for_seed(y,seed)
    severity_table=severity_counts_for_split(y,train_idx,val_idx,test_idx)
    print(f'Seed {seed}: Severity counts in each partition')
    display(severity_table)
    severity_table.to_csv(run_dir/'split_severity_counts.csv')
    print(f'Run {run_number}/5 seed {seed}: train={len(train_idx)}, validation={len(val_idx)}, test={len(test_idx)}',flush=True)
    manifest={'seed':seed,'run':run_number,'counts':{},'index_sha256':{},'class_counts':{}}
    for part,idx in [('train',train_idx),('validation',val_idx),('test',test_idx)]:
        manifest['counts'][part]=len(idx)
        manifest['index_sha256'][part]=hashlib.sha256(np.sort(row_ids[idx]).tobytes()).hexdigest()
        manifest['class_counts'][part]={str(classes[c]):int((y[idx]==c).sum()) for c in range(n_classes)}
    (run_dir/'split_manifest.json').write_text(json.dumps(manifest,indent=2))
    if SAVE_SPLIT_INDICES:np.savez_compressed(run_dir/'split_row_ids.npz',train=row_ids[train_idx],validation=row_ids[val_idx],test=row_ids[test_idx])
    Xtr=X.iloc[train_idx];Xva=X.iloc[val_idx];Xte=X.iloc[test_idx]
    ytr,yva,yte=y[train_idx],y[val_idx],y[test_idx]
    # Determine category groups from the training partition only.
    numeric_cols=[c for c in Xtr if pd.api.types.is_numeric_dtype(Xtr[c])]
    cat_cols=[c for c in Xtr if c not in numeric_cols]
    high_card=[c for c in cat_cols if Xtr[c].nunique()>MAX_CAT_ONEHOT]
    low_card=[c for c in cat_cols if c not in high_card]
    Xtr_p,Xrest_p,pre=preprocess_ml(Xtr,ytr,pd.concat([Xva,Xte]),seed=seed)
    Xva_p,Xte_p=Xrest_p[:len(Xva)],Xrest_p[len(Xva):]
    del Xtr,Xva,Xte;gc.collect()
    if SAVE_MODELS:joblib.dump(pre,run_dir/'preprocess.pkl')
    sw=class_weights(ytr)
    majority=int(pd.Series(ytr).mode()[0])
    for name in MODEL_NAMES:
        model=make_models(seed=seed)[name]
        started=time.perf_counter()
        model.fit(Xtr_p,ytr,sample_weight=sw)
        fit_seconds=time.perf_counter()-started
        val_proba=np.asarray(model.predict_proba(Xva_p))
        thresholds=tune_thresholds(yva,val_proba,list(range(n_classes)))
        (run_dir/f'{name}_thresholds.json').write_text(json.dumps({str(classes[k]):float(v) for k,v in thresholds.items()},indent=2))
        del val_proba
        test_proba=np.asarray(model.predict_proba(Xte_p))
        before=test_proba.argmax(axis=1)
        after=predict_with_thresholds(test_proba,thresholds,list(range(n_classes)),majority)
        scores={}
        for stage,pred in [('Before',before),('After',after)]:
            scores[stage]=calc_metrics(yte,pred)
            metric_rows.append({'Run':run_number,'Seed':seed,'Model':name,'Stage':stage,'Fit_seconds':fit_seconds,**scores[stage]})
            report_predictions(name,stage,yte,pred,run_dir,seed,run_number)
        delta_rows.append({'Run':run_number,'Seed':seed,'Model':name,
            **{m:scores['After'][m]-scores['Before'][m] for m in METRICS}})
        if SAVE_TEST_PREDICTIONS:
            pd.DataFrame({'Row_ID':row_ids[test_idx],'y_true':np.asarray(classes)[yte],
              'Before':np.asarray(classes)[before],'After':np.asarray(classes)[after]}).to_csv(run_dir/f'{name}_test_predictions.csv',index=False)
        if SAVE_TEST_PROBABILITIES:np.savez_compressed(run_dir/f'{name}_test_probabilities.npz',row_ids=row_ids[test_idx],classes=np.asarray(classes),probabilities=test_proba)
        if SAVE_MODELS:joblib.dump(model,run_dir/f'{name}.pkl')
        print(name,'Before:',scores['Before'],'After:',scores['After'],flush=True)
        flush_results()
        del test_proba,before,after,model;gc.collect()
    (run_dir/'run_status.json').write_text(json.dumps({'status':'complete','seed':seed}))
    del Xtr_p,Xrest_p,Xva_p,Xte_p,pre,sw;gc.collect()
assert len(metric_rows)==30 and len(delta_rows)==15
print('All five runs completed.')


## Mean ± standard deviation tables and charts
Before/after metrics are aggregated separately. Tuning impact uses each run’s paired after-minus-before difference. Positive changes indicate improvement. Standard deviations use ddof=1 across five runs. Per-class support is saved per run; it is not an independent count across overlapping test sets.

In [ ]:
metrics_df=pd.DataFrame(metric_rows);deltas_df=pd.DataFrame(delta_rows);per_class_df=pd.DataFrame(per_class_rows)

def save_mean_std(frame,groupby,metrics,stem):
    summary=frame.groupby(groupby)[metrics].agg(['mean','std'])
    flat=summary.copy();flat.columns=[f'{m}_{s}' for m,s in flat.columns]
    flat.to_csv(OUT_DIR/f'{stem}_numeric.csv')
    formatted=pd.DataFrame(index=summary.index)
    for m in metrics:
        formatted[m]=[f'{a:.4f} ± {b:.4f}' for a,b in zip(summary[(m,'mean')],summary[(m,'std')])]
    formatted.to_csv(OUT_DIR/f'{stem}_mean_std.csv')
    display(formatted)
    return summary

print('Before tuning: test mean ± std')
before_summary=save_mean_std(metrics_df[metrics_df.Stage=='Before'],'Model',METRICS,'before_test')
print('After tuning: test mean ± std')
after_summary=save_mean_std(metrics_df[metrics_df.Stage=='After'],'Model',METRICS,'after_test')
print('Paired impact (after − before): mean ± std')
delta_summary=save_mean_std(deltas_df,'Model',METRICS,'threshold_impact')
class_summary=save_mean_std(per_class_df,['Model','Stage','Severity'],['Precision','Recall','F1'],'per_class')
class_delta=per_class_df.pivot(index=['Run','Seed','Model','Severity'],columns='Stage',values=['Precision','Recall','F1'])
class_delta=pd.DataFrame({m:class_delta[(m,'After')]-class_delta[(m,'Before')] for m in ['Precision','Recall','F1']}).reset_index()
class_delta.to_csv(OUT_DIR/'run_per_class_deltas.csv',index=False)
class_delta_summary=save_mean_std(class_delta,['Model','Severity'],['Precision','Recall','F1'],'per_class_impact')

x=np.arange(len(MODEL_NAMES));width=.36
fig,axes=plt.subplots(2,2,figsize=(12,8))
for ax,m in zip(axes.flat,METRICS):
    for offset,summary,label,color in [(-width/2,before_summary,'Before','#4477AA'),(width/2,after_summary,'After','#EE6677')]:
        table=summary.reindex(MODEL_NAMES)
        ax.bar(x+offset,table[(m,'mean')],width,yerr=table[(m,'std')],capsize=4,label=label,color=color)
    ax.set_xticks(x,MODEL_NAMES);ax.set_ylim(0,1);ax.set_ylabel('Test score');ax.set_title(m.replace('_',' '));ax.legend();ax.grid(axis='y',alpha=.2)
fig.suptitle('Five repeated splits: test mean ± sample std')
fig.tight_layout();fig.savefig(OUT_DIR/'before_after_mean_std.png',dpi=300);fig.savefig(OUT_DIR/'before_after_mean_std.pdf');plt.show()
fig,axes=plt.subplots(2,2,figsize=(12,8))
for ax,m in zip(axes.flat,METRICS):
    table=delta_summary.reindex(MODEL_NAMES)
    ax.bar(x,table[(m,'mean')],yerr=table[(m,'std')],capsize=4,color='#228833')
    ax.axhline(0,color='black',linewidth=.8);ax.set_xticks(x,MODEL_NAMES)
    ax.set_title(m.replace('_',' '));ax.set_ylabel('After − before');ax.grid(axis='y',alpha=.2)
fig.suptitle('Threshold tuning: paired change mean ± sample std')
fig.tight_layout();fig.savefig(OUT_DIR/'threshold_impact_mean_std.png',dpi=300);plt.show()
fig,axes=plt.subplots(1,3,figsize=(15,4.8),sharey=True)
for ax,name in zip(axes,MODEL_NAMES):
    for stage,color in [('Before','#4477AA'),('After','#EE6677')]:
        rows=metrics_df[(metrics_df.Model==name)&(metrics_df.Stage==stage)].sort_values('Run')
        ax.plot(rows.Run,rows.Macro_F1,'o-',label=stage,color=color)
    ax.set_title(name);ax.set_xticks(range(1,6));ax.set_xlabel('Run');ax.set_ylim(0,1);ax.legend();ax.grid(alpha=.2)
axes[0].set_ylabel('Test Macro-F1');fig.tight_layout();fig.savefig(OUT_DIR/'macro_f1_by_run.png',dpi=300);plt.show()
fig,axes=plt.subplots(1,3,figsize=(15,5),sharey=True)
for ax,name in zip(axes,MODEL_NAMES):
    for offset,stage,color in [(-width/2,'Before','#4477AA'),(width/2,'After','#EE6677')]:
        part=class_summary.loc[(name,stage)].reindex(classes)
        ax.bar(np.arange(len(classes))+offset,part[('F1','mean')],width,yerr=part[('F1','std')],capsize=3,label=stage,color=color)
    ax.set_title(name);ax.set_xticks(np.arange(len(classes)),classes);ax.set_xlabel('Severity');ax.set_ylim(0,1);ax.legend()
axes[0].set_ylabel('Per-class test F1');fig.suptitle('Per-severity F1: five-run mean ± std')
fig.tight_layout();fig.savefig(OUT_DIR/'per_severity_f1_mean_std.png',dpi=300);plt.show()
(OUT_DIR/'run_status.txt').write_text('SUCCESS: all five seeds and three models completed.\n')
archive=shutil.make_archive(str(OUT_DIR),'zip',OUT_DIR)
print('Download results:',archive);display(FileLink(archive))
